In [1]:
import os
DATA_PATH = './data/'
videos = [DATA_PATH + f for f in os.listdir(DATA_PATH) if f.endswith('.mp4')]

videos

['./data/1.mp4', './data/2.mp4']

In [2]:
from nbutils.video_utils import show

show(videos[0])

QFontDatabase: Cannot find font directory /home/kendama/Projects/Personal/HackYeah26/AirTriage_HY26_App/.venv/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/kendama/Projects/Personal/HackYeah26/AirTriage_HY26_App/.venv/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/kendama/Projects/Personal/HackYeah26/AirTriage_HY26_App/.venv/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/kendama/Projects/Personal/HackYeah26/AirTriage_HY26_App/.venv/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no 

In [ ]:
import os
os.environ.setdefault("QT_LOGGING_RULES", "qt.qpa.*=false")  # hides Qt font warnings

import subprocess, time
from pathlib import Path

import cv2
from IPython.display import display, clear_output, Image, Video


def get_device():
    """Best available accelerator: CUDA, Apple Silicon, or CPU."""
    try:
        import torch
        if torch.cuda.is_available():
            return 0
        if torch.backends.mps.is_available():
            return "mps"
    except ImportError:
        pass
    return "cpu"


def read_frames(path):
    cap = cv2.VideoCapture(str(path))
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            yield frame
    finally:
        cap.release()


def play(frames, fps=25):
    """Show BGR frames inline as JPEGs. No browser codecs, no GUI."""
    delay = 1 / fps
    for frame in frames:
        t = time.time()
        ok, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
        clear_output(wait=True)
        display(Image(data=buf.tobytes()))
        time.sleep(max(0, delay - (time.time() - t)))


def preview(path, width=640):
    """Make a small WebM copy (plays in every browser) and embed it."""
    import imageio_ffmpeg
    path = Path(path)
    out = path.with_name(path.stem + "_preview.webm")
    subprocess.run([
        imageio_ffmpeg.get_ffmpeg_exe(), "-y", "-loglevel", "error", "-i", str(path),
        "-c:v", "libvpx-vp9", "-b:v", "0", "-crf", "35", "-an", str(out)
    ], check=True)
    return Video(str(out), embed=True, mimetype="video/webm", width=width)

In [4]:
import cv2
from ultralytics import YOLO

model = YOLO("yolov8m.pt")  # n = fastest; s/m/l/x = more accurate, slower

cap = cv2.VideoCapture(str(videos[0]))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 25

out = cv2.VideoWriter("detected.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

while True:
    ok, frame = cap.read()
    if not ok:
        break

    result = r = model(
    frame,
    classes=[0],
    conf=0.4,      # default 0.25; lower = more detections, more false positives
    iou=0.8,        # NMS overlap threshold (default 0.7); higher keeps overlapping people in crowds
    imgsz=1280,     # default 640; helps a lot with small or distant people
    max_det=300,
    verbose=False
)[0]

    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"person {conf:.2f}", (x1, max(y1 - 6, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    cv2.putText(frame, f"count: {len(result.boxes)}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    out.write(frame)

cap.release()
out.release()